# residual-skip-add composite — cx26: residual-skip-add wrapping an nn.Sequential main path

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `residual-skip-add`, `module-composition`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "residual-skip-add"
DD_ATOM_IDS = ["residual-skip-add", "module-composition"]
DD_SUBTOPICS = ["CNN: Residual skip-connection add", "PyTorch: Module composition"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Real ResNet blocks are often refactored so the **main path** is one `nn.Sequential` and the **skip-add** is the only thing the outer module's `forward` does explicitly. This isolates the architectural choice (the layer stack) from the residual wiring.

**Module composition** is PyTorch's term for *building bigger modules out of smaller ones*. Three idioms show up:
- `nn.Sequential(*layers)` — a list-shaped composition; forward is `layers[0](layers[1](...))`.
- subclass with named children — flexible; you write the forward.
- mixed — outer subclass holds an inner `nn.Sequential` for the linear stretch and adds a skip on top.

The third idiom is what cx26 exercises: a residual block whose `main` attribute IS an `nn.Sequential(conv1, bn1, relu, conv2, bn2)` (no final relu inside it!), and whose `forward` is just `F.relu(self.main(x) + x)`.

**Anatomy.**
```python
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
            nn.ReLU(),
            nn.Conv2d(c, c, 3, padding=1, bias=False),
            nn.BatchNorm2d(c),
        )
    def forward(self, x):
        return F.relu(self.main(x) + x)        # skip-add inside the wrapper.
```

**Why care.** Mixing `nn.Sequential` for the linear stretch with an outer subclass for the skip is the textbook PyTorch composition pattern. It also makes the main path a single object you can swap out (e.g. bottleneck variant) without touching the skip wiring.

### Composite Exercise — residual-skip-add wrapping an nn.Sequential main path

**Atoms exercised together**: `residual-skip-add`, `module-composition`

Implement `cx26_make_resblock_with_sequential()` — return the class.

Required structure for the returned class `ResBlock(nn.Module)`:
- `__init__(self, channels)` calls `super().__init__()` then sets:
  - `self.main = nn.Sequential(...)` containing, in order:
    1. `nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)`
    2. `nn.BatchNorm2d(channels)`
    3. `nn.ReLU()`
    4. `nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)`
    5. `nn.BatchNorm2d(channels)`
  - **No final ReLU inside `self.main`** — that goes in `forward` after the skip-add.
- `forward(self, x)` returns `F.relu(self.main(x) + x)`.

The test verifies: (a) `self.main` is an `nn.Sequential` of length 5; (b) the skip-add is load-bearing (zeroing the main-path weights yields `relu(x)`).

In [ ]:
def cx26_make_resblock_with_sequential():
    class ResBlock(nn.Module):
        def __init__(self, channels):
            super().__init__()
            # Atom B (module-composition): the linear stretch lives inside an nn.Sequential.
            # NOTE: no final ReLU inside self.main — that's in forward AFTER the skip-add.
            self.main = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(channels),
            )

        def forward(self, x):
            # Atom A (residual-skip-add): main path + skip, then final ReLU.
            return F.relu(self.main(x) + x)

    return ResBlock


<details><summary>Show solution — cx26</summary>

```python
def cx26_make_resblock_with_sequential():
    class ResBlock(nn.Module):
        def __init__(self, channels):
            super().__init__()
            # Atom B (module-composition): the linear stretch lives inside an nn.Sequential.
            # NOTE: no final ReLU inside self.main — that's in forward AFTER the skip-add.
            self.main = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(),
                nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(channels),
            )

        def forward(self, x):
            # Atom A (residual-skip-add): main path + skip, then final ReLU.
            return F.relu(self.main(x) + x)

    return ResBlock
```

The two atoms split the responsibility cleanly: `self.main` (composition) is the architecture knob — swap it for a bottleneck variant and everything else still works. The skip-add lives in `forward` because the outer module is the only place where the input `x` and the main-path output `self.main(x)` co-exist.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx26'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx26',
        'subtopics': ["CNN: Residual skip-connection add", "PyTorch: Module composition"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()